In [5]:
import requests
import json

URL='https://ac.search.naver.com/nx/ac'
headers={
    'User-Agent':
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36 Edg/151.0.0.0'
}

In [6]:
def get_related_keywords(keyword: str) -> list[str]:
    """키워드의 연관검색어를 리스트로 돌려준다. 없으면 빈 리스트."""
    try:
        r = requests.get(URL, params={"q": keyword, "_callback": "_jsonp_7", 'q_enc': 'UTF-8', 'st': '100'},
                         headers=headers, timeout=10)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"요청 실패: {e}")
        return []
    
    result = json.loads(r.text.split('(', maxsplit=1)[-1][:-1])
    return [item[0].strip() for item in result['items'][0]]

In [7]:
if __name__ == "__main__":
    for kw in ["부트캠프", "웹크롤링", "ㅁㄴㅇㄹ"]:
        result = get_related_keywords(kw)
        print(f"{kw:8s} → {len(result)}건 {result[:5]}")

부트캠프     → 10건 ['부트캠프', '부트캠프 뜻', '부트캠프 취업', 'ai 부트캠프', '직무부트캠프']
웹크롤링     → 5건 ['웹크롤링', '웹 크롤링 뜻', '웹 크롤링 자동화', '웹 크롤링 프로그램', '웹 크롤링 방법']
ㅁㄴㅇㄹ     → 10건 ['메니에르병', '메니에르 증상', '매니에르병', '메니에르', '물놀이 렌즈']


In [8]:
import requests
import pandas as pd

URL='https://comic.naver.com/api/webtoon/titlelist/weekday'
DETAIL='https://comic.naver.com/webtoon/list?titleId='
HEADERS={
    'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36 Edg/151.0.0.0'
}
DAY_KR = {
    "MONDAY": "월", "TUESDAY": "화", "WEDNESDAY": "수", "THURSDAY": "목",
    "FRIDAY": "금", "SATURDAY": "토", "SUNDAY": "일", "DAILY_PLUS": "매일+",
}

In [10]:
def fetch() -> list[dict]:
    res = requests.get(URL, headers=HEADERS, timeout=10)
    res.raise_for_status()
    data = res.json()

    rows = []
    for key, toons in data.get("titleListMap", {}).items():
        day = DAY_KR.get(key, key)
        for t in toons:
            rows.append({
                "제목": t.get("titleName", ""),
                "링크": DETAIL + str(t.get("titleId", "")),
                "요일": day,
            })
    return rows

In [11]:
rows=fetch()

df=pd.DataFrame(rows).drop_duplicates(subset=['링크','요일'])
df.to_csv('naver.webtoon.csv',index=False,encoding='cp949')
print(f'{len(df)}건 수집 - 요일 분포\n{df['요일'].value_counts()}')

775건 수집 - 요일 분포
요일
토    121
금    116
화    113
월    112
수    107
목    106
일    100
Name: count, dtype: int64


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup